#Clustering: California Housing


## 1. Análisis del dataset

El dataset *California Housing Prices* contiene datos del **censo de California de 1990**, donde cada fila representa un **bloque censal**. Se usa para predecir el valor mediano de las viviendas.

| Variable | Descripción |
|---|---|
| `longitude` / `latitude` | Ubicación geográfica del bloque |
| `housing_median_age` | Antigüedad mediana de las viviendas |
| `total_rooms` / `total_bedrooms` | Totales de habitaciones y dormitorios del bloque |
| `population` | Población total del bloque |
| `households` | Cantidad de hogares |
| `median_income` | Ingreso mediano (en decenas de miles de USD) |
| `median_house_value` | **Target.** Valor mediano de las viviendas en USD |
| `ocean_proximity` | Cercanía al océano (categórica) |

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
import kagglehub

path = kagglehub.dataset_download('camnugent/california-housing-prices')
df = pd.read_csv(path + '/housing.csv')
print(df.shape)
display(df.head())
print(df.isnull().sum())
display(df.describe())

In [ ]:
# Histogramas
df.hist(figsize=(12, 8), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('Distribución de variables', fontweight='bold')
plt.tight_layout()
plt.show()

## Preprocesamiento

- **Imputación:** `total_bedrooms` tiene ~207 nulos → se imputa con la mediana.
- **Ratios:** `total_rooms`, `total_bedrooms`, `population` y `households` son totales absolutos que dependen del tamaño del bloque. Se reemplazan por ratios más representativos: `rooms_per_household`, `bedrooms_per_room`, `population_per_household`.
- **Codificación:** `ocean_proximity` se convierte a numérica con `LabelEncoder`.
- **Estandarización:** K-Means y DBSCAN son sensibles a la escala → `StandardScaler`.

In [ ]:
df2 = df.copy()
df2['total_bedrooms'] = SimpleImputer(strategy='median').fit_transform(df2[['total_bedrooms']])
df2['rooms_per_household']      = df2['total_rooms']    / df2['households']
df2['bedrooms_per_room']        = df2['total_bedrooms'] / df2['total_rooms']
df2['population_per_household'] = df2['population']     / df2['households']
df2['ocean_enc'] = LabelEncoder().fit_transform(df2['ocean_proximity'])

features = ['longitude','latitude','housing_median_age','median_income',
            'median_house_value','rooms_per_household','bedrooms_per_room',
            'population_per_household','ocean_enc']
X = StandardScaler().fit_transform(df2[features])
print('Shape listo para clustering:', X.shape)

## 2. K-Means
### 2.1 Selección de k

Se usan **tres métricas complementarias**:
- **Inercia (WCSS):** suma de distancias al centroide → buscar el codo.
- **Silhouette Score:** qué tan bien separado está cada punto de los clusters vecinos. Rango [-1,1], mayor es mejor.
- **Davies-Bouldin Index:** razón entre dispersión interna y separación entre clusters → menor es mejor.

Usar solo una métrica puede ser engañoso: el codo indica cuándo agregar más clusters deja de aportar, Silhouette confirma cohesión interna y DB valida la separación.

In [ ]:
k_range = range(2, 11)
inertias, sil, db = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil.append(silhouette_score(X, labels, sample_size=5000, random_state=42))
    db.append(davies_bouldin_score(X, labels))

ks = list(k_range)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, vals, title, color in zip(axes,
    [inertias, sil, db],
    ['Inercia (codo)', 'Silhouette (↑)', 'Davies-Bouldin (↓)'],
    ['steelblue', 'seagreen', 'tomato']):
    ax.plot(ks, vals, 'o-', color=color)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('k')
    ax.grid(alpha=0.3)
plt.suptitle('Métricas para selección de k', fontweight='bold')
plt.tight_layout()
plt.show()

print('Mejor k por Silhouette:', ks[np.argmax(sil)])
print('Mejor k por DB Index: ', ks[np.argmin(db)])

**Elección: k=4.** El codo aparece en k=3 o k=4, Silhouette y DB también apuntan a ese rango. k=4 ofrece el mejor balance entre separación estadística e interpretabilidad (zonas costeras premium, suburbios, interior rural, zonas densas urbanas).

In [ ]:
K = 4
km = KMeans(n_clusters=K, init='k-means++', n_init=15, random_state=42)
df2['cluster_km'] = km.fit_predict(X)

print('Distribución:', df2['cluster_km'].value_counts().sort_index().to_dict())
print('Silhouette final:', silhouette_score(X, df2['cluster_km'], sample_size=5000, random_state=42).round(4))

### 2.2 Análisis descriptivo por cluster

In [ ]:
vars_desc = ['median_house_value','median_income','housing_median_age',
             'rooms_per_household','population_per_household']
display(df2.groupby('cluster_km')[vars_desc].mean().round(2))
print()
display(pd.crosstab(df2['cluster_km'], df2['ocean_proximity'], normalize='index').round(2))

In [ ]:
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
fig, axes = plt.subplots(1, len(vars_desc), figsize=(16, 5))
for ax, var in zip(axes, vars_desc):
    data = [df2.loc[df2['cluster_km']==c, var].values for c in range(K)]
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col); patch.set_alpha(0.75)
    ax.set_xticklabels([f'C{i}' for i in range(K)])
    ax.set_title(var, fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Diagramas de caja por cluster — K-Means (k=4)', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación de clusters:**

| Cluster | Perfil |
|---|---|
| 0 | Zona costera premium: mayor valor e ingreso, cerca del océano |
| 1 | Interior rural: bajo valor e ingreso, predominio INLAND |
| 2 | Suburbios / 1h del océano: valores intermedios |
| 3 | Zonas densas urbanas: alta población por hogar, ingreso bajo-medio |

## 3. DBSCAN

**DBSCAN** agrupa puntos por densidad sin necesitar k predefinido. Los puntos en zonas de baja densidad se etiquetan como **ruido (-1)**. Sus hiperparámetros son:
- `eps`: radio de vecindad.
- `min_samples`: mínimo de puntos para ser *core point*.

Se usa la **curva k-distance** para estimar eps: el codo en la curva indica el valor adecuado.

In [ ]:
# Features reducidas para DBSCAN (geoeconómicas continuas)
X_db = StandardScaler().fit_transform(
    df2[['longitude','latitude','median_income','median_house_value']]
)

MIN_SAMPLES = 10
dists, _ = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_db).kneighbors(X_db)
k_dist = np.sort(dists[:, -1])[::-1]

plt.figure(figsize=(9, 3))
plt.plot(k_dist, linewidth=1, color='steelblue')
plt.axhline(0.35, color='red', linestyle='--', label='eps ≈ 0.35')
plt.xlabel('Puntos'); plt.ylabel('Distancia al 10° vecino')
plt.title('Curva k-distance para selección de eps', fontweight='bold')
plt.ylim(0, 2); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
db_model = DBSCAN(eps=0.35, min_samples=MIN_SAMPLES)
df2['cluster_db'] = db_model.fit_predict(X_db)

n_clusters = len(set(df2['cluster_db'])) - (1 if -1 in df2['cluster_db'].values else 0)
n_noise = (df2['cluster_db'] == -1).sum()
print(f'Clusters: {n_clusters} | Ruido: {n_noise} ({100*n_noise/len(df2):.1f}%)')
print(df2['cluster_db'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mapa clusters
labels_uniq = sorted(df2['cluster_db'].unique())
cmap = plt.cm.get_cmap('tab10', max(len(labels_uniq)-1, 1))
for lbl in labels_uniq:
    mask = df2['cluster_db'] == lbl
    color = '#cccccc' if lbl == -1 else cmap(lbl)
    axes[0].scatter(df2.loc[mask,'longitude'], df2.loc[mask,'latitude'],
                    s=2, alpha=0.4, color=color, label='Ruido' if lbl==-1 else f'C{lbl}')
axes[0].set_title('DBSCAN — Clusters', fontweight='bold')
axes[0].legend(markerscale=5, fontsize=7, ncol=2)

# Mapa outliers
noise = df2['cluster_db'] == -1
axes[1].scatter(df2.loc[~noise,'longitude'], df2.loc[~noise,'latitude'], s=1, alpha=0.3, color='lightblue')
axes[1].scatter(df2.loc[noise,'longitude'], df2.loc[noise,'latitude'], s=8, color='red', alpha=0.7, label=f'Ruido ({n_noise})')
axes[1].set_title('DBSCAN — Outliers', fontweight='bold')
axes[1].legend(markerscale=2)

plt.tight_layout(); plt.show()

# Boxplots DBSCAN
df_clean = df2[df2['cluster_db'] != -1]
clusters_ids = sorted(df_clean['cluster_db'].unique())
vars_db = ['median_house_value','median_income','housing_median_age']
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, var in zip(axes, vars_db):
    data = [df_clean.loc[df_clean['cluster_db']==c, var].values for c in clusters_ids]
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    cmap2 = plt.cm.get_cmap('tab10', len(clusters_ids))
    for patch, c in zip(bp['boxes'], clusters_ids):
        patch.set_facecolor(cmap2(c)); patch.set_alpha(0.75)
    ax.set_xticklabels([f'C{c}' for c in clusters_ids], fontsize=8)
    ax.set_title(var, fontsize=9, fontweight='bold'); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Diagramas de caja por cluster — DBSCAN', fontweight='bold')
plt.tight_layout(); plt.show()

**Interpretación de DBSCAN:**

- Los **clusters principales** corresponden a las zonas metropolitanas densas (Bay Area, LA, San Diego): alta cohesión geográfica y económica.
- Los **puntos de ruido** representan bloques aislados: zonas rurales remotas o bloques con valores extremos de ingreso/vivienda que no pertenecen a ninguna región densa.
- A diferencia de K-Means, DBSCAN **no fuerza la asignación** de todos los puntos, lo que permite detectar outliers reales del mercado inmobiliario.
- Los clusters siguen formas arbitrarias (costas, valles), lo que K-Means con centroides no puede capturar fielmente.

---
## Conclusiones

- **K-Means (k=4):** segmenta el mercado en 4 perfiles interpretables (costas premium, suburbios, interior rural, zonas densas urbanas). Las tres métricas (inercia, Silhouette, DB) convergieron en ese rango.
- **DBSCAN:** detecta la estructura de densidad real del territorio y separa como ruido los bloques atípicos. Complementa a K-Means para identificar anomalías genuinas.
- El preprocesamiento (ratios, imputación, escalado) fue esencial para que ambos algoritmos funcionen correctamente.

## Referencias

- Nugent, C. (2018). *California Housing Prices*. Kaggle. https://www.kaggle.com/datasets/camnugent/california-housing-prices
- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
- Ester et al. (1996). A density-based algorithm for discovering clusters. *KDD*, 96(34), 226–231.
- Rousseeuw, P. J. (1987). Silhouettes. *Journal of Computational and Applied Mathematics*, 20, 53–65.